# Day 3: CLI tool (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — str methods, format specs, bytes vs str.

In [ ]:
k, _, v = "key=value=more".partition("=")

In [ ]:
assert (k, v) == ("key", "value=more")

In [ ]:
assert f"{3.14159:.2f}|{42:>5}|{255:#x}|{1234567:,}" == "3.14|   42|0xff|1,234,567"
assert "é".encode("utf-8") == b"\xc3\xa9" and b"\xc3\xa9".decode() == "é"
assert "a,b,,c".split(",") == ["a", "b", "", "c"]
assert "  x ".strip() == "x" and "abc"[::-1] == "cba"

**Then, vary it:**

- VARIATION: use rpartition and rsplit(maxsplit=1)
- VARIATION: format a table row with f"{name:<10}{qty:>5}"

## `s2_flow.py`: Flow — match on command shapes, sentinel loop.

In [ ]:
def run(cmd):
    match cmd.split():
        case ["add", x, y]: return int(x) + int(y)
        case ["echo", *rest]: return " ".join(rest)
        case ["quit"]: return None
        case _: return "?"

In [ ]:
assert run("add 2 3") == 5 and run("echo a b") == "a b"
assert run("quit") is None and run("zzz") == "?"

In [ ]:
lines = ["a", "", "b"]

In [ ]:
assert [(i, l) for i, l in enumerate(lines, 1) if l] == [(1, "a"), (3, "b")]

In [ ]:
it, total = iter([1, 2, 3]), 0
while (x := next(it, None)) is not None: total += x

In [ ]:
assert total == 6

**Then, vary it:**

- VARIATION: add a case ["mul", x, y] and a guard rejecting non-digits
- VARIATION: match a dict command {"op": "add", "args": [..]}

## `s3_functions.py`: Functions — decorator factory, generator function, multi-return.

In [ ]:
import functools
def repeat(n):
    def deco(fn):
        @functools.wraps(fn)
        def w(*a, **k): return [fn(*a, **k) for _ in range(n)]
        return w
    return deco
@repeat(3)
def hi(): return "hi"

In [ ]:
assert hi() == ["hi"] * 3

In [ ]:
def read_nonblank(lines):
    for l in lines:
        if l.strip(): yield l.strip()

In [ ]:
assert list(read_nonblank(["a", " ", "b "])) == ["a", "b"]

In [ ]:
def minmax(xs): return min(xs), max(xs)
lo, hi_ = minmax([3, 1, 2])

In [ ]:
assert (lo, hi_) == (1, 3)

**Then, vary it:**

- VARIATION: make read_nonblank accept a comment prefix to skip
- VARIATION: stack two decorators and predict the order they apply

## `s4_classes.py`: Classes — Protocol + registry, callable objects.

In [ ]:
from typing import Protocol
class Command(Protocol):
    def run(self, arg: str) -> str: ...
class Upper:
    def run(self, arg): return arg.upper()
class Rev:
    def run(self, arg): return arg[::-1]
REGISTRY: dict[str, Command] = {"upper": Upper(), "rev": Rev()}
def dispatch(name, arg): return REGISTRY[name].run(arg)

In [ ]:
assert dispatch("upper", "ab") == "AB" and dispatch("rev", "ab") == "ba"

In [ ]:
class Adder:
    def __init__(self, n): self.n = n
    def __call__(self, x): return x + self.n

In [ ]:
assert Adder(2)(3) == 5

**Then, vary it:**

- VARIATION: register commands with a decorator @command("name")
- VARIATION: add a Title command; no base class needed (structural typing)

## `s5_errors.py`: Errors/IO — argparse, pathlib globbing, missing files.

In [ ]:
import argparse, tempfile
from pathlib import Path
p = argparse.ArgumentParser()
p.add_argument("path")
p.add_argument("-n", type=int, default=2)
p.add_argument("-v", "--verbose", action="store_true")
a = p.parse_args(["f.txt", "-n", "5", "-v"])

In [ ]:
assert (a.path, a.n, a.verbose) == ("f.txt", 5, True)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "sub").mkdir()
    (root / "sub" / "a.py").write_text("x\n")
    (root / "b.txt").write_text("y\n")
    assert [f.name for f in root.rglob("*.py")] == ["a.py"]
    try: (root / "nope").read_text()
    except FileNotFoundError: pass
    else: raise AssertionError
    assert (root / "b.txt").stat().st_size == 2

**Then, vary it:**

- VARIATION: add a subcommand with add_subparsers
- VARIATION: use Path.suffix/.stem/.with_suffix on "a/b.tar.gz"

## `s6_stdlib.py`: Stdlib — re named groups, shlex, subprocess, textwrap.

In [ ]:
import re, shlex, subprocess, sys, textwrap
m = re.match(r"(?P<lvl>ERROR|INFO): (?P<msg>.*)", "ERROR: disk full")

In [ ]:
assert m["lvl"] == "ERROR" and m["msg"] == "disk full"

In [ ]:
assert re.sub(r"\d+", "#", "a1b22") == "a#b#"
assert shlex.split('a "b c" d') == ["a", "b c", "d"]
r = subprocess.run([sys.executable, "-c", "print('hi')"], capture_output=True, text=True, check=True)

In [ ]:
assert r.stdout.strip() == "hi"

In [ ]:
assert textwrap.dedent("  a\n  b") == "a\nb"

**Then, vary it:**

- VARIATION: compile the regex with re.VERBOSE and comments
- VARIATION: make subprocess fail; catch CalledProcessError and read .returncode

## `s7_concurrency.py`: Concurrency — process pool for CPU work.

In [ ]:
%%writefile _pool_worker.py
def sq(n): return n * n

In [ ]:
import importlib; importlib.invalidate_caches()
from concurrent.futures import ProcessPoolExecutor
from _pool_worker import sq

with ProcessPoolExecutor(2) as ex:
    r = list(ex.map(sq, range(5)))

In [ ]:
assert r == [0, 1, 4, 9, 16]

**Then, vary it:**

Notebook note: process-pool workers must live in an importable module (spawned children can't see functions defined in cells), so the worker is written with `%%writefile`.
- VARIATION: delete the __main__ guard and read the error (macOS/Windows spawn)
- VARIATION: swap in ThreadPoolExecutor; same result, different cost model

## `s8_quality.py`: Quality — logging to a handler, tracebacks.

In [ ]:
import logging, traceback
class ListHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.msgs = []
    def emit(self, r): self.msgs.append(r.getMessage())
log = logging.getLogger("t")
log.setLevel(logging.INFO)
h = ListHandler()
log.addHandler(h)
log.info("x=%d", 3)
log.debug("skip")

In [ ]:
assert h.msgs == ["x=3"]

In [ ]:
try: 1 / 0
except ZeroDivisionError: assert "ZeroDivisionError" in traceback.format_exc()

**Then, vary it:**

- VARIATION: set level to DEBUG and assert both messages
- VARIATION: run this file with `python -m pdb` and step to the try block

## `kata.py`: KATA — mini grep.

In [ ]:
import argparse, re, tempfile
from pathlib import Path
def main(argv):
    p = argparse.ArgumentParser()
    p.add_argument("pattern")
    p.add_argument("path")
    p.add_argument("-i", action="store_true")
    a = p.parse_args(argv)
    flags = re.I if a.i else 0
    return [l for l in Path(a.path).read_text().splitlines() if re.search(a.pattern, l, flags)]

In [ ]:
from pathlib import Path
import argparse, re, tempfile

with tempfile.TemporaryDirectory() as d:
    f = Path(d) / "x.txt"
    f.write_text("Foo\nbar\nfoo bar\n")
    assert main(["foo", str(f)]) == ["foo bar"]
    assert main(["-i", "foo", str(f)]) == ["Foo", "foo bar"]

**Then, vary it:**

- KATA: add -n (line numbers) and -c (count only) flags